<!-- journey-badges -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NolaNexus/LMp2LMa/blob/main/track-04-ml-architect/12_profiling.ipynb)
[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/NolaNexus/LMp2LMa/blob/main/track-04-ml-architect/12_profiling.ipynb)
[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/NolaNexus/LMp2LMa)

# 12 — Profiling and Performance Engineering

**Objectives**
- Measure before optimizing: wall-clock, throughput, and memory.
- Use `torch.profiler` to find the real hotspot.
- Estimate parameters/FLOPs to reason about capacity before benchmarks.


In [ ]:
import os
import time

import torch
import torch.nn as nn

torch.manual_seed(0)
QUICK_MODE = os.getenv("QUICK_MODE", "0") == "1"

model = nn.Sequential(
    nn.Linear(512, 1024),
    nn.ReLU(),
    nn.Linear(1024, 1024),
    nn.ReLU(),
    nn.Linear(1024, 10),
)
x = torch.randn(64, 512)

steps = 5 if QUICK_MODE else 50
start = time.perf_counter()
for _ in range(steps):
    loss = model(x).sum()
    loss.backward()
    model.zero_grad()
elapsed = time.perf_counter() - start
print(f"steps: {steps}")
print(f"per_step_ms: {elapsed / steps * 1000:.2f}")
print(f"throughput_samples_per_s: {steps * x.shape[0] / elapsed:.0f}")

In [ ]:
# Where does the time actually go?
from torch.profiler import ProfilerActivity, profile

with profile(activities=[ProfilerActivity.CPU], record_shapes=True) as prof:
    for _ in range(3):
        model(x).sum().backward()
        model.zero_grad()

print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=8))

In [ ]:
params = sum(p.numel() for p in model.parameters())
flops_per_sample = 2 * sum(
    layer.in_features * layer.out_features for layer in model if isinstance(layer, nn.Linear)
)
print(f"parameters: {params:,}")
print(f"flops_per_sample: {flops_per_sample:,}")
print(f"memory_params_mb: {params * 4 / 1024**2:.2f}")

**Method**
1. Baseline: wall-clock + throughput at a realistic batch size.
2. Profile: sort by CPU/CUDA time; ignore noise below a few percent.
3. Optimize the top item, then re-measure. Keep a record of before/after.

Never tune on a single run — warm-up and variance will fool you.
